# NHANES Data Analysis & Obesity Prediction
## 04 — Logistic Regression & Assumption Verification

## This notebook fits a logistic regression model to predict obesity and verifies the key assumptions required for the model to be valid.

## 1. Load the Dataset and Select Features

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score

# Load the dataset
filepath = r"C:\Users\Roza Aissaoui\Desktop\Dashboard project\Code\nhanes_obesity_analysis\clean_merged_data.csv"
df = pd.read_csv(filepath)

# Select the features and target 
X = df[['RIAGENDR', 'RIDAGEYR', 'RIDRETH3', 'DMDEDUC2', 'DMDMARTZ', 
        'INDFMPIR', 'BMXWAIST', 'BMXARMC', 'BMXHIP', 'BMXHT']]
Y = df['Obese']

## 2. Train/Test Split

We split the data into 70% training and 30% testing to evaluate model performance on unseen data.

In [4]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, random_state=1
)

# Combine X_train and y_train into a single DataFrame 
#train_data = pd.concat([X_train, Y_train], axis=1)
#test_data = pd.concat([X_test, Y_test], axis=1)

# Save the train/test datasets
#train_path = r"C:\Users\Roza Aissaoui\Desktop\Dashboard project\Code\nhanes_obesity_analysis\train_dataset.csv"
#test_path = r"C:\Users\Roza Aissaoui\Desktop\Dashboard project\Code\nhanes_obesity_analysis\test_dataset.csv"

#train_data.to_csv(train_path, index=False)
#test_data.to_csv(test_path, index=False)
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
Y_train.to_csv("Y_train.csv", index=False)
Y_test.to_csv("Y_test.csv", index=False)

## 3. Fit the Initial Logistic Regression Model

We use statsmodels to obtain a full statistical summary including coefficients, p‑values, AIC, and BIC.

In [8]:
# Import the training dataset

file_path = r"C:\Users\Roza Aissaoui\Desktop\Dashboard project\Code\nhanes_obesity_analysis\X_train.csv"
df_X_train = pd.read_csv(file_path)

file_path1 = r"C:\Users\Roza Aissaoui\Desktop\Dashboard project\Code\nhanes_obesity_analysis\Y_train.csv"
df_Y_train = pd.read_csv(file_path1)

# Add constant (intercept)
X_train_const = sm.add_constant(X_train)

# Fit logistic regression
logit_model = sm.Logit(Y_train, X_train_const).fit()

# Model summary
print(logit_model.summary())
print("AIC:", logit_model.aic)
print("BIC:", logit_model.bic)

Optimization terminated successfully.
         Current function value: 0.168446
         Iterations 10
                           Logit Regression Results                           
Dep. Variable:                  Obese   No. Observations:                10517
Model:                          Logit   Df Residuals:                    10506
Method:                           MLE   Df Model:                           10
Date:                Mon, 08 Jun 2026   Pseudo R-squ.:                  0.7517
Time:                        10:53:41   Log-Likelihood:                -1771.5
converged:                       True   LL-Null:                       -7133.8
Covariance Type:            nonrobust   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -22.5308      1.263    -17.844      0.000     -25.006     -20.056
RIAGENDR      -1.2620      0

### The logistic regression model identifies several significant predictors of obesity. Body‑measurement variables (waist, hip, arm circumference) show strong positive associations with obesity, as expected clinically. Some demographic variables are not statistically significant, indicating they do not meaningfully contribute to predicting obesity in this model. The model fits well overall, with a high pseudo R² and reasonable AIC/BIC values.

## 4. Backward Stepwise Feature Elimination

We remove variables with p‑values > 0.05 to obtain a more parsimonious model.

In [9]:
def backward_stepwise(X, Y, threshold=0.05):
    X = X.copy()
    while True:
        model = sm.Logit(Y, X).fit(disp=0)
        p_values = model.pvalues.drop("const", errors='ignore')
        max_p = p_values.max()
        if max_p > threshold:
            worst_feature = p_values.idxmax()
            print(f"Removing '{worst_feature}' with p-value {max_p:.4f}")
            X = X.drop(columns=worst_feature)
        else:
            break
    return X, model

# Run backward elimination
X_optimized, logit_model_optimized = backward_stepwise(X_train_const, Y_train)

# Final optimized model summary
print(logit_model_optimized.summary())
print("AIC:", logit_model_optimized.aic)
print("BIC:", logit_model_optimized.bic)

Removing 'DMDMARTZ' with p-value 0.9071
Removing 'INDFMPIR' with p-value 0.8072
Removing 'RIDRETH3' with p-value 0.6853
Removing 'DMDEDUC2' with p-value 0.2097
                           Logit Regression Results                           
Dep. Variable:                  Obese   No. Observations:                10517
Model:                          Logit   Df Residuals:                    10510
Method:                           MLE   Df Model:                            6
Date:                Mon, 08 Jun 2026   Pseudo R-squ.:                  0.7515
Time:                        10:53:52   Log-Likelihood:                -1772.4
converged:                       True   LL-Null:                       -7133.8
Covariance Type:            nonrobust   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -22.3742      1.254    -17.836      0

### In the optimized model, only body‑measurement variables remain significant predictors of obesity. Waist, hip, and arm circumference show strong positive associations, while demographic variables drop out due to lack of significance. The model maintains a strong fit, indicating that physical measurements are the primary drivers of obesity prediction in this dataset.

## 5. Model Evaluation on Training Data

We evaluate the optimized model using a confusion matrix and accuracy score.

In [10]:
# Predict probabilities
Y_train_pred_prob = logit_model_optimized.predict(X_optimized)

# Convert probabilities to class labels
Y_pred_class = (Y_train_pred_prob >= 0.5).astype(int)

# Confusion matrix and accuracy
conf_matrix_train = confusion_matrix(Y_train, Y_pred_class)
accuracy_train = accuracy_score(Y_train, Y_pred_class)

print("Confusion Matrix (Training Data):\n", conf_matrix_train)
print("\nAccuracy (Training Data):", accuracy_train)

Confusion Matrix (Training Data):
 [[5855  307]
 [ 369 3986]]

Accuracy (Training Data): 0.9357231149567367


### The model performs very well on the training data, correctly classifying most cases with high accuracy. Misclassifications are relatively low, indicating strong in‑sample predictive ability.

## 6. Model Evaluation on Test Data

We apply the optimized model to the test dataset to assess generalization.

In [11]:
# Load test dataset
file_path = r"C:\Users\Roza Aissaoui\Desktop\Dashboard project\Code\nhanes_obesity_analysis\X_test.csv"
df_X_test = pd.read_csv(file_path)

file_path1 = r"C:\Users\Roza Aissaoui\Desktop\Dashboard project\Code\nhanes_obesity_analysis\Y_test.csv"
df_X_test = pd.read_csv(file_path1)

# Keep only optimized model features
X_test_reduced = X_test[X_optimized.columns.drop("const", errors='ignore')]

# Add constant
X_test_const = sm.add_constant(X_test_reduced)

# Predict probabilities
Y_test_pred_prob = logit_model_optimized.predict(X_test_const)

# Convert to class predictions
Y_pred_class = (Y_test_pred_prob >= 0.5).astype(int)

# Confusion matrix and accuracy
conf_matrix_test = confusion_matrix(Y_test, Y_pred_class)
accuracy_test = accuracy_score(Y_test, Y_pred_class)

print("Confusion Matrix (Test Data):\n", conf_matrix_test)
print("\nAccuracy (Test Data):", accuracy_test)

Confusion Matrix (Test Data):
 [[2541  110]
 [ 150 1707]]

Accuracy (Test Data): 0.9423247559893523


### The model performs very well on the test data, showing high accuracy and low misclassification rates. This indicates strong generalization and confirms that the model predicts obesity reliably on test data.

## 7. Assumption Verification

### 7.1. Binary Outcome Variable

The dependent variable must contain only two categories.

In [12]:
print(Y_train.unique())

[0 1]


### This confirms that the outcome variable is binary, which satisfies the first assumption of logistic regression.

### 7.2 Independence of Observations

In [13]:
filepath = r"C:\Users\Roza Aissaoui\Desktop\Dashboard project\Code\nhanes_obesity_analysis\clean_merged_data.csv"
df = pd.read_csv(filepath)
duplicate_rows = df[df.duplicated()]
print(duplicate_rows)

Empty DataFrame
Columns: [SEQN, RIAGENDR, RIDAGEYR, RIDRETH3, DMDEDUC2, DMDMARTZ, INDFMPIR, BMXBMI, BMXWAIST, BMXARMC, BMXHIP, BMXHT, Obese]
Index: []


### 7.3 Linearity of Log-Odds
Previously checked in 03_eda_part2_visualization. 

### 7.4 No Multicollinearity (VIF)

In [14]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data["Feature"] = X_test_const.columns
vif_data["VIF"] = [
    variance_inflation_factor(X_test_const.values, i)
    for i in range(X_test_const.shape[1])
]

print(vif_data)

    Feature         VIF
0     const  733.210990
1  RIAGENDR    2.538976
2  RIDAGEYR    1.287968
3  BMXWAIST    5.717917
4   BMXARMC    4.118035
5    BMXHIP    6.210061
6     BMXHT    2.029990


### The VIF values are all below common concern thresholds, indicating no serious multicollinearity among the predictors.

## 8. Notebook Summary

### - Logistic regression model successfully fitted  
### - Backward elimination removed non‑significant predictors  
### - Final model kept only significant body‑measurement variables  
### - Strong accuracy on both training and test sets  
### - Outcome variable is binary 
### - No duplicate observations 
### - Linearity of log‑odds previously verified 
### - VIF values acceptable, no multicollinearity   
### - Assumptions sufficiently met for logistic regression  